In [1]:
import io
import time
import os
import numpy as np
from huggingface_hub import upload_file

In [5]:
repo_ix = "veerlosar/prism-model-activations"
repo_type = "dataset"
#hf_folder_path = "meta-llama/Llama-3.1-8B-Instruct/adherence_gpt_mini_activations"
hf_folder_path = "Qwen/Qwen3-8B/adherence_gpt_mini_activations"

In [6]:
REMOTE_DIR = "/workspace/probing/probes_data"
probing_dir = "/workspace/probing"

In [4]:
!pwd

/home/masha/Desktop/Projects/PRISM/crosslingual-rule-following/canonical/probing


In [ ]:
for lang in ['en', 'de', 'it', 'yo', 'hi', 'ig', 'ru', 'ur']:
    filename_X = f"X_adh_{lang}.npy"
    filename_y = f"y_adh_{lang}.npy"
    ready_flag = f"X_adh_{lang}.ready"

    while True:
        # Check if the .ready file exists on the remote machine
        check_file = !ssh runpod-direct "test -f {probing_dir}/{ready_flag} && echo 'YES' || echo 'NO'"
        
        if "YES" in check_file:
            break
        time.sleep(5)
        
    print(f"{lang} is ready! Pulling to local machine...")
    
    # pulling the files
    !scp runpod-direct:{REMOTE_DIR}/{filename_X} probes_data/{filename_X}
    !scp runpod-direct:{REMOTE_DIR}/{filename_y} probes_data/{filename_y}
    
    # delete on runpod
    !ssh runpod-direct "rm {REMOTE_DIR}/{filename_X} {probing_dir}/{ready_flag}"
    !ssh runpod-direct "rm {REMOTE_DIR}/{filename_y}"
    
    print(f"Files cleared from remote. Processing local upload for batch {lang}...")

    
    upload_file(
        path_or_fileobj=f"probes_data/{filename_X}",
        path_in_repo=f"{hf_folder_path}/{filename_X}",
        repo_id=repo_ix,
        repo_type=repo_type,
    )
    upload_file(
        path_or_fileobj=f"probes_data/{filename_y}",
        path_in_repo=f"{hf_folder_path}/{filename_y}",
        repo_id=repo_ix,
        repo_type=repo_type,
    )
    
    # remove locally
    os.remove(f"probes_data/{filename_X}")
    os.remove(f"probes_data/{filename_y}")
    print(f"{lang} successfully uploaded to Hugging Face!\n")

en is ready! Pulling to local machine...
X_adh_en.npy                                   57% 4562MB   5.9MB/s   09:26 ETA